# IA causal aplicada a manufactura con DoWhy

**Notebook listo para Google Colab — versión en español**  
**Caso:** efecto de reducir la velocidad de una línea sobre la tasa de defectos.

## Objetivo

Una planta observa que algunas órdenes se producen a menor velocidad y quiere saber si esa intervención **causa** una reducción en los defectos, o si la diferencia observada se explica por que las máquinas más antiguas, los turnos con mayor temperatura o las órdenes con mantenimiento pendiente reciben un tratamiento distinto.

La pregunta causal es:

> ¿Cuál es el efecto promedio de reducir la velocidad de producción sobre la tasa de defectos, manteniendo comparables las condiciones que influyen tanto en la intervención como en el resultado?

El notebook recorre el flujo de DoWhy: **modelar → identificar → estimar → refutar**.

> Los datos son sintéticos. No representan una planta real y no deben usarse para decisiones operativas sin sustituirlos por datos observacionales validados.


## Compatibilidad actual

Al momento de preparar este notebook, la versión estable publicada en PyPI es **DoWhy 0.14**. El repositorio oficial indica instalación con `pip install dowhy` y soporte para Python 3.8 o superior. Para hacer el ejercicio reproducible en Colab, se fija `dowhy==0.14` y se imprime la versión instalada.

Fuentes oficiales:

- [DoWhy en PyPI](https://pypi.org/project/dowhy/)
- [Repositorio oficial de DoWhy](https://github.com/py-why/dowhy)


In [ ]:
# Instalación reproducible para Google Colab.
# Si se desea siempre la versión estable más reciente, sustituir por: pip install -q dowhy
!pip -q install dowhy==0.14


In [ ]:
import sys
import importlib.metadata as metadata

print(f"Python: {sys.version.split()[0]}")
print(f"DoWhy: {metadata.version('dowhy')}")


## 1. Preparar el entorno

Se usan `numpy` y `pandas` para generar y manipular datos, `matplotlib` y `seaborn` para exploración visual, y `DoWhy` para formular y estimar la pregunta causal.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dowhy import CausalModel

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)


## 2. Generar un dataset sintético reproducible

Cada fila representa una orden de producción. La variable `reduccion_velocidad` es la intervención: 1 significa que la línea operó temporalmente 10% más lento. La variable objetivo `tasa_defectos` mide defectos por cada 100 unidades.

Para representar confusión, la intervención no se asigna al azar: es más probable cuando hay máquinas antiguas, alta temperatura, mantenimiento pendiente o baja experiencia del operador. Esos mismos factores también afectan los defectos.

El efecto causal verdadero incorporado en el generador es aproximadamente **−1.5 defectos por cada 100 unidades** cuando se reduce la velocidad, antes de ruido aleatorio. DoWhy debe recuperar un efecto cercano al verdadero al ajustar por los confundidores observados.


In [ ]:
n = 1500

machine_age = rng.uniform(0, 15, n)                 # años
ambient_temp = rng.normal(24, 4, n).clip(12, 38)   # °C
operator_experience = rng.normal(5, 2, n).clip(0.5, 12)  # años
maintenance_due = rng.binomial(1, 0.25, n)         # 1: mantenimiento vencido

# Riesgo operativo previo que influye en la decisión de reducir velocidad.
logit_treatment = (
    -1.0 + 0.18 * machine_age + 0.12 * (ambient_temp - 24)
    - 0.30 * operator_experience + 1.10 * maintenance_due
)
prob_treatment = 1 / (1 + np.exp(-logit_treatment))
reduccion_velocidad = rng.binomial(1, prob_treatment)

# Modelo estructural del resultado: el coeficiente -1.5 es el efecto causal diseñado.
tasa_defectos = (
    2.2
    + 0.22 * machine_age
    + 0.16 * (ambient_temp - 24)
    - 0.20 * operator_experience
    + 1.40 * maintenance_due
    - 1.50 * reduccion_velocidad
    + rng.normal(0, 1.15, n)
).clip(0.05, None)

df = pd.DataFrame({
    "machine_age": machine_age,
    "ambient_temp": ambient_temp,
    "operator_experience": operator_experience,
    "maintenance_due": maintenance_due,
    "reduccion_velocidad": reduccion_velocidad,
    "tasa_defectos": tasa_defectos,
})

print(f"Filas: {len(df):,}")
display(df.head())


### Explicación detallada del bloque de generación

1. `SEED` controla el generador aleatorio. Si se vuelve a ejecutar el notebook con la misma semilla, se obtiene la misma muestra y los mismos resultados.
2. `machine_age`, `ambient_temp`, `operator_experience` y `maintenance_due` representan condiciones observables del proceso.
3. `logit_treatment` convierte esas condiciones en una propensión a reducir la velocidad. La función logística transforma cualquier puntuación en una probabilidad entre 0 y 1.
4. `reduccion_velocidad` se genera como una variable binaria a partir de esa probabilidad. Por eso existe confusión intencional: el tratamiento no se asigna de manera completamente aleatoria.
5. `tasa_defectos` es una ecuación estructural. Cada coeficiente indica cómo cambia el resultado esperado cuando cambia una variable y las demás se mantienen constantes. El término `-1.50` es la señal causal que se desea recuperar.
6. El ruido representa variación no explicada: materia prima, microparadas, medición, lote y otros factores no incluidos.

Este diseño permite enseñar el problema central de la inferencia causal: una diferencia entre grupos puede combinar el efecto de la intervención con diferencias preexistentes entre los grupos.


## 3. Validar los datos y comparar grupos

La diferencia simple entre órdenes tratadas y no tratadas es una asociación. Puede estar sesgada porque ambos grupos no tienen la misma mezcla de antigüedad, temperatura, experiencia y mantenimiento.


In [ ]:
print("Valores faltantes:")
display(df.isna().sum().to_frame("faltantes").T)

resumen_grupos = df.groupby("reduccion_velocidad")["tasa_defectos"].agg(["count", "mean", "std"]).rename(
    index={0: "Sin reducción", 1: "Con reducción"}
)
display(resumen_grupos.round(3))

diferencia_ingenua = (
    df.loc[df.reduccion_velocidad == 1, "tasa_defectos"].mean()
    - df.loc[df.reduccion_velocidad == 0, "tasa_defectos"].mean()
)
print(f"Diferencia ingenua (tratados - no tratados): {diferencia_ingenua:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="tasa_defectos", hue="reduccion_velocidad", kde=True, ax=axes[0], bins=30)
axes[0].set_title("Distribución de la tasa de defectos")
axes[0].set_xlabel("Defectos por cada 100 unidades")
axes[0].legend(title="Reducción", labels=["Sin reducción", "Con reducción"])

sns.boxplot(data=df, x="reduccion_velocidad", y="tasa_defectos", ax=axes[1])
axes[1].set_title("Resultado por grupo")
axes[1].set_xlabel("Reducción de velocidad (0/1)")
axes[1].set_ylabel("Tasa de defectos")
plt.tight_layout()
plt.show()


### Cómo leer el análisis exploratorio

La tabla de grupos muestra cuántas órdenes recibieron cada condición y el promedio de defectos en cada grupo. La resta de promedios es útil como primer diagnóstico, pero no es todavía una estimación causal.

La visualización ayuda a identificar tres aspectos:

- **Dirección:** si el grupo tratado presenta menos defectos, la asociación es compatible con una mejora, pero no demuestra que la reducción sea la causa.
- **Dispersión:** una variabilidad alta puede hacer que el promedio sea inestable; por eso conviene revisar intervalos de confianza y tamaño de muestra.
- **Comparabilidad:** si los grupos tienen composiciones muy diferentes, la comparación directa puede estar sesgada. En este ejercicio esa falta de comparabilidad fue creada intencionalmente mediante la regla de asignación del tratamiento.

La inferencia causal intenta responder una comparación distinta: qué tasa de defectos habría tenido la misma orden bajo `reduccion_velocidad=1` frente a `reduccion_velocidad=0`. Esa segunda situación es contrafactual y nunca se observa simultáneamente para la misma orden.


## 4. Definir el DAG causal

El grafo dirigido acíclico (DAG) expresa las hipótesis del proceso:

- La antigüedad, temperatura, experiencia y mantenimiento influyen en la decisión de reducir velocidad.
- Esas variables también influyen en la tasa de defectos.
- La reducción de velocidad tiene un efecto directo sobre la tasa de defectos.

El conjunto de variables de ajuste es, por tanto, `{machine_age, ambient_temp, operator_experience, maintenance_due}`. No se ajustan variables posteriores a la intervención.


In [ ]:
import networkx as nx

dag_edges = [
    ("machine_age", "reduccion_velocidad"), ("machine_age", "tasa_defectos"),
    ("ambient_temp", "reduccion_velocidad"), ("ambient_temp", "tasa_defectos"),
    ("operator_experience", "reduccion_velocidad"), ("operator_experience", "tasa_defectos"),
    ("maintenance_due", "reduccion_velocidad"), ("maintenance_due", "tasa_defectos"),
    ("reduccion_velocidad", "tasa_defectos"),
]
graph = nx.DiGraph(dag_edges)
positions = {
    "machine_age": (-1.2, 1.0), "ambient_temp": (-1.2, 0.3),
    "operator_experience": (-1.2, -0.4), "maintenance_due": (-1.2, -1.1),
    "reduccion_velocidad": (0.2, 0.0), "tasa_defectos": (1.5, 0.0),
}
plt.figure(figsize=(12, 5))
nx.draw_networkx(graph, pos=positions, node_color="#cfe8ff", node_size=2600,
                 arrows=True, arrowsize=20, font_size=9, edge_color="#4a5568")
plt.title("DAG: reducción de velocidad y tasa de defectos")
plt.axis("off")
plt.show()


In [ ]:
dag_dot = """
digraph {
    machine_age -> reduccion_velocidad;
    machine_age -> tasa_defectos;
    ambient_temp -> reduccion_velocidad;
    ambient_temp -> tasa_defectos;
    operator_experience -> reduccion_velocidad;
    operator_experience -> tasa_defectos;
    maintenance_due -> reduccion_velocidad;
    maintenance_due -> tasa_defectos;
    reduccion_velocidad -> tasa_defectos;
}
"""

model = CausalModel(
    data=df,
    treatment="reduccion_velocidad",
    outcome="tasa_defectos",
    graph=dag_dot,
    proceed_when_unidentifiable=True,
)
identified_estimand = model.identify_effect()
print(identified_estimand)


### Explicación detallada del DAG y de la identificación

El DAG no es un dibujo decorativo: es la representación explícita de los supuestos del análisis.

- Las flechas hacia `reduccion_velocidad` indican variables que pueden influir en la decisión de bajar la velocidad.
- Las flechas hacia `tasa_defectos` indican variables que pueden modificar la calidad.
- La flecha `reduccion_velocidad → tasa_defectos` representa la relación que queremos estimar.

Si, por ejemplo, `machine_age` afecta tanto al tratamiento como al resultado, existe un camino no causal `reduccion_velocidad ← machine_age → tasa_defectos`. Ajustar por `machine_age` bloquea ese camino. DoWhy usa el criterio de puerta trasera para identificar un conjunto suficiente de variables de ajuste.

Una advertencia importante: el DAG no descubre por sí solo la verdad del proceso. Si se omite una causa relevante —por ejemplo, tipo de material—, la estimación puede conservar confusión residual. Si se ajusta por un mediador posterior o por un colisionador, también puede introducirse sesgo.


## 5. Estimar el efecto causal

Se usa regresión lineal como estimador de puerta trasera (`backdoor.linear_regression`). En este ejemplo, DoWhy identifica el conjunto de confundidores a partir del DAG y estima el efecto de cambiar la intervención de 0 a 1.

La unidad del resultado es **defectos por cada 100 unidades**. Un valor negativo significa que reducir la velocidad disminuye los defectos.


In [ ]:
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True,
    test_significance=True,
)

print(causal_estimate)
print(f"\nEfecto causal estimado: {causal_estimate.value:.3f} defectos por cada 100 unidades")


### Qué hace cada parte del bloque de estimación

- `model.identify_effect()` traduce el DAG en un estimando causal y determina qué caminos deben bloquearse.
- `model.estimate_effect(...)` elige un método estadístico para calcular ese estimando a partir de los datos.
- `backdoor.linear_regression` ajusta una regresión con el tratamiento y las variables de confusión identificadas.
- `confidence_intervals=True` solicita un intervalo de incertidumbre cuando el estimador lo soporta.
- `test_significance=True` solicita una prueba de significancia estadística.

El número reportado es un **ATE** (Average Treatment Effect): el cambio promedio esperado en la tasa de defectos si toda la población operara con reducción de velocidad frente a si nadie la utilizara, bajo los supuestos del DAG.

La unidad es importante: `-1.5` no significa una reducción de 1.5% en la velocidad ni una reducción relativa del 1.5%. Significa aproximadamente 1.5 defectos menos por cada 100 unidades producidas. Para convertirlo a impacto económico habría que multiplicar por volumen, costo de scrap, costo de retrabajo y pérdida de throughput.


### Interpretación de la estimación

Compara el efecto causal estimado con el efecto verdadero usado al generar los datos (`-1.5`). En datos reales, no conoceríamos ese valor: lo usamos aquí únicamente para comprobar que el procedimiento didáctico recupera aproximadamente la señal incorporada.

La diferencia ingenua puede alejarse de `-1.5` porque las órdenes con mayor riesgo tienen una probabilidad distinta de recibir la reducción de velocidad. La estimación ajustada intenta comparar unidades con condiciones operativas similares.


In [ ]:
efecto_verdadero = -1.5
error_absoluto = abs(causal_estimate.value - efecto_verdadero)
print(f"Efecto verdadero conocido por construcción: {efecto_verdadero:.3f}")
print(f"Error absoluto de recuperación: {error_absoluto:.3f}")


### Cómo interpretar una diferencia entre la estimación y `-1.5`

La estimación no tiene que ser exactamente igual al valor verdadero porque la muestra incluye ruido y tiene un tamaño finito. Una diferencia pequeña es esperable por variación muestral. Una diferencia grande puede indicar, entre otras causas:

1. Confusión no controlada o variables relevantes ausentes del DAG.
2. Falta de soporte común: algunas condiciones aparecen casi exclusivamente en un grupo.
3. Una relación no lineal que la regresión lineal no representa bien.
4. Interacciones, por ejemplo que la reducción funcione distinto en máquinas antiguas.
5. Error de medición o una definición inestable del tratamiento y del resultado.

Por eso el valor puntual nunca debe leerse aislado. Conviene observar el intervalo de confianza, el tamaño de muestra, los diagnósticos de datos y la estabilidad ante especificaciones alternativas.


## 6. Pruebas de refutación

Las pruebas de refutación no prueban que el DAG sea verdadero. Evalúan si la estimación es sensible a cambios que, bajo los supuestos del análisis, no deberían alterar sustancialmente la conclusión.

1. **Causa común aleatoria:** agrega una causa común aleatoria; el efecto no debería cambiar de forma relevante.
2. **Tratamiento placebo:** reemplaza la intervención por una asignación aleatoria; el efecto debería acercarse a cero.


In [ ]:
refutation_random = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name="random_common_cause",
)
print("Refutación: causa común aleatoria")
print(refutation_random)


In [ ]:
refutation_placebo = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
)
print("Refutación: tratamiento placebo")
print(refutation_placebo)


### Lectura detallada de las refutaciones

La prueba de causa común aleatoria introduce una variable que no debería tener relación real con la decisión ni con los defectos. Si el estimador cambia poco, aumenta la confianza en que no está reaccionando de manera exagerada a una variable irrelevante. Si cambia mucho, hay que investigar inestabilidad o sensibilidad del modelo.

La prueba placebo permuta la asignación del tratamiento. Como esa nueva asignación no contiene la señal causal original, su efecto debería acercarse a cero. Un placebo claramente distinto de cero puede sugerir problemas en la especificación, dependencia entre observaciones, estructura de datos o un resultado numéricamente inestable.

Estas pruebas son controles de robustez, no una certificación de causalidad. No detectan automáticamente todos los confundidores no observados ni sustituyen un experimento o una validación de proceso.


## 7. Interpretar los resultados automáticamente

El siguiente bloque traduce los resultados a lenguaje de negocio. Los umbrales son didácticos: en una aplicación real deben definirse con ingeniería de calidad, costos de scrap, capacidad, seguridad y límites de proceso.


In [ ]:
efecto = float(causal_estimate.value)
print("Resumen ejecutivo")
if efecto < 0:
    print(f"• La estimación sugiere una reducción de {abs(efecto):.2f} defectos por cada 100 unidades al reducir la velocidad.")
else:
    print(f"• La estimación no sugiere una reducción: el efecto estimado es {efecto:.2f} defectos por cada 100 unidades.")

print("• Esta conclusión depende de que el DAG incluya los confundidores relevantes y de que no haya sesgo importante por medición o selección.")
print("• El placebo debe producir un efecto cercano a cero; si no ocurre, revisar la especificación del modelo y los datos.")


### De la estimación estadística a una decisión de manufactura

Una estimación negativa puede justificar una prueba operativa, pero no implica que se deba reducir la velocidad en todas las líneas. La decisión debe separar al menos cuatro preguntas:

- **Calidad:** ¿cuántos defectos se evitan y en qué productos o máquinas?
- **Productividad:** ¿cuántas unidades adicionales dejan de producirse por hora?
- **Economía:** ¿el ahorro por defectos evitados supera el costo de menor producción?
- **Operación y seguridad:** ¿la intervención respeta límites de equipo, contrato, seguridad y calidad?

El ATE resume un promedio. Es posible que el efecto sea más fuerte en máquinas antiguas o cuando el mantenimiento está vencido. Para una implementación real convendría estimar efectos heterogéneos por segmento y comprobar que cada segmento tenga suficiente soporte de datos.


## 8. Conclusiones y recomendaciones

### Conclusiones del ejercicio

- La diferencia observada entre grupos no es automáticamente causal.
- El DAG hace explícitos los supuestos sobre el proceso de manufactura y permite identificar variables de ajuste.
- DoWhy separa identificación, estimación y refutación, lo que facilita auditar el razonamiento.
- En este dataset sintético, el efecto ajustado debe aproximarse al efecto diseñado de `-1.5` defectos por cada 100 unidades, con variación por muestreo.

### Recomendaciones para una planta real

1. Sustituir el dataset sintético por datos por orden, máquina, turno y lote; conservar fecha y trazabilidad.
2. Validar con ingeniería qué variables son causas, cuáles son mediadores y cuáles son colisionadores; no ajustar por variables posteriores a la reducción de velocidad.
3. Registrar explícitamente la regla por la que se decide reducir velocidad; esa política determina el riesgo de confusión.
4. Complementar el análisis observacional con una prueba controlada o un despliegue escalonado cuando sea seguro y viable.
5. Convertir el efecto en una decisión económica: costo de menor throughput frente a ahorro por menor scrap y retrabajo.
6. Repetir las refutaciones, revisar sensibilidad a confundidores no observados y monitorear el efecto por máquina, producto y turno.

### Limitaciones

El resultado no demuestra que reducir velocidad funcione en una planta real. La validez externa, la calidad de medición, la interferencia entre órdenes y la presencia de causas no observadas requieren validación adicional.


### Conclusión ampliada

El aprendizaje principal es metodológico: la pregunta relevante no es si las órdenes tratadas tienen menos defectos, sino qué habría ocurrido con las mismas condiciones operativas bajo la alternativa de tratamiento. DoWhy obliga a declarar esas condiciones mediante un DAG, identificar los caminos no causales y someter el resultado a refutaciones.

En este ejemplo, la reducción de velocidad fue diseñada para tener un efecto favorable y los confundidores fueron observados. Por ello el análisis debe recuperar un efecto cercano a `-1.5` defectos por cada 100 unidades. En una aplicación real, el resultado tendría que acompañarse de:

1. Un diccionario de datos y reglas de calidad.
2. Evidencia de que la política de asignación del tratamiento está bien documentada.
3. Análisis de sensibilidad a confundidores no observados.
4. Comparaciones por máquina, producto, turno, lote y periodo.
5. Un piloto controlado o escalonado con métricas de calidad y productividad.

La recomendación práctica no es simplemente “bajar la velocidad”, sino diseñar una política de intervención verificable: cuándo aplicar la reducción, durante cuánto tiempo, a qué equipos, con qué límites y cómo revertirla si el throughput o la calidad empeoran.


## 9. Checklist para adaptar el notebook

- [ ] Reemplazar la generación sintética por `pd.read_csv(...)` o una consulta autorizada.
- [ ] Confirmar que tratamiento y resultado tienen una definición operacional estable.
- [ ] Dibujar el DAG con expertos del proceso antes de estimar.
- [ ] Verificar datos faltantes, valores extremos y soporte común entre tratados y controles.
- [ ] Ejecutar al menos una refutación y documentar qué supuesto evalúa.
- [ ] Revisar el efecto por segmentos antes de recomendar un cambio de set point.
